# SahelPay — Partie 1 : cluster Spark et positionnement

**Objectif de la partie** : démarrer un cluster Spark, s'y connecter depuis ce notebook, et comprendre qui fait quoi.

**Ce qu'on fait dans ce notebook**
1. Comprendre l'architecture Spark (Driver, Cluster Manager, Executors).
2. Créer une `SparkSession` connectée au cluster.
3. Lancer un petit calcul pour voir le cluster travailler.
4. Vérifier que Spark sait parler à **MinIO** (stockage) et **Kafka** (flux), dont on aura besoin dans les parties suivantes.

## 1. L'architecture de Spark

```
   Ton notebook (DRIVER)                 CLUSTER
 ┌─────────────────────┐        ┌───────────────────────┐
 │  SparkSession       │ ─────► │ Spark Master          │  ← Cluster Manager
 │  (SparkContext)     │        │ (spark-master:7077)   │
 └─────────────────────┘        └──────────┬────────────┘
           ▲                               │ alloue des ressources
           │ résultats                     ▼
           │              ┌─────────────────┐  ┌─────────────────┐
           └──────────────│ Worker 1        │  │ Worker 2        │
                          │  └ Executor     │  │  └ Executor     │
                          │    (2 cœurs)    │  │    (2 cœurs)    │
                          └─────────────────┘  └─────────────────┘
```

| Élément | Rôle | Dans notre projet Docker |
|---|---|---|
| **Driver** | Le programme principal. Il découpe le travail en *tâches* et récupère les résultats. | le conteneur `jupyter` (ce notebook) |
| **SparkSession / SparkContext** | Le point d'entrée : c'est par lui qu'on parle au cluster. | l'objet `spark` / `sc` créé plus bas |
| **Cluster Manager** | Décide quelles ressources (cœurs, mémoire) sont données à chaque application. | le conteneur `spark-master` |
| **Worker** | Une machine du cluster qui héberge des executors. | `spark-worker-1` et `spark-worker-2` |
| **Executor** | Un processus qui exécute les tâches et garde les données en mémoire. | lancés sur les workers à la demande |

**Lien avec Hadoop** : le Cluster Manager joue le rôle du *ResourceManager* YARN, les Workers celui des *NodeManagers*, et les Executors celui des *containers*.

## 2. Pourquoi Spark plutôt que MapReduce ?

Deux limites de MapReduce que Spark corrige, et qui comptent pour SahelPay :

1. **Les écritures disque entre chaque étape.** MapReduce écrit sur HDFS entre le *map* et le *reduce*. Spark garde les données **en mémoire** entre les étapes, donc les enchaînements (filtrer → transformer → agréger) sont beaucoup plus rapides.
2. **Pas de temps réel.** MapReduce ne fait que du *batch*. Spark offre le **Structured Streaming** : on pourra lire les transactions de Kafka au fil de l'eau (Parties 4 et 5).

## 3. Créer la SparkSession

Le fichier `scripts/spark_session.py` construit la session pour nous. Il configure :
- **le Master** à contacter : `spark://spark-master:7077` ;
- **les ressources demandées** : 4 cœurs au maximum (2 workers × 2 cœurs), 1 Go par executor ;
- **les librairies** : connecteur Kafka, accès S3 (pour MinIO) et Delta Lake ;
- **MinIO** : l'adresse du serveur et les identifiants, pour écrire dans `s3a://lake/...`.

> ⏳ **Le premier lancement est long** : Spark télécharge les librairies depuis Internet. Ne relance pas la cellule tant qu'elle affiche `[*]`.

In [1]:
# --- 1. Rendre le module du projet importable ---------------------------------
# /workspace = ton dépôt Git (monté dans le conteneur par docker-compose).
import sys
sys.path.append("/workspace/scripts")      # pour que Python trouve spark_session.py

from spark_session import get_spark        # notre fonction qui construit la SparkSession

# --- 2. Créer la SparkSession -------------------------------------------------
# Ce notebook devient le DRIVER : il se connecte au Master et lui demande des ressources.
# "SahelPay-Partie1" = nom de l'application, visible dans l'interface http://localhost:8080
spark = get_spark("SahelPay-Partie1")

# --- 3. Récupérer le SparkContext ---------------------------------------------
# C'est l'objet "bas niveau" de Spark : il sert à créer des RDD (Partie 2).
sc = spark.sparkContext

# --- 4. Afficher les informations demandées par le sujet ----------------------
print("Version Spark :", spark.version)           # version de Spark
print("Master        :", sc.master)               # adresse du Cluster Manager
print("appName       :", sc.appName)              # nom de l'application
print("Cœurs         :", sc.defaultParallelism)   # parallélisme par défaut (voir note plus bas)

ModuleNotFoundError: No module named 'spark_session'

### Lire le résultat

- **Version Spark : 3.5.3** → c'est la version demandée.
- **Master : `spark://spark-master:7077`** → on est bien connecté à notre cluster et pas à un Spark local.
- **appName** → le nom qu'on a choisi ; on le retrouve dans l'interface du Master (http://localhost:8080).
- **Cœurs : 2** → c'est normal, et ce n'est pas la valeur finale. Au moment de l'affichage, les executors ne sont pas encore démarrés sur les workers : Spark donne alors une valeur minimale de 2. On le re-vérifiera après le premier calcul, où on doit voir **4** (2 workers × 2 cœurs).

## 4. Un premier calcul distribué

On crée un **RDD** (une collection répartie sur le cluster) contenant les nombres de 0 à 999 999, puis on calcule leur somme.

Trois notions à retenir :
- **Partition** : un morceau du RDD. Une partition = une **tâche** pour un cœur.
- **Transformation vs action** : `parallelize` ne calcule rien (c'est une description du travail). C'est l'**action** `sum()` qui déclenche vraiment l'exécution. On appelle ça l'*évaluation paresseuse* (lazy evaluation).
- **Stage** : un groupe de tâches exécutables en parallèle sans échange de données entre partitions.

In [ ]:
# Un RDD (Resilient Distributed Dataset) = collection répartie en partitions.
# range(1_000_000) : les entiers de 0 à 999 999.
# numSlices=4 : on force 4 partitions => 4 tâches, une par cœur disponible (2 workers x 2 cœurs).
rdd = sc.parallelize(range(1_000_000), numSlices=4)

# sum() est une ACTION : c'est elle qui lance le calcul sur les executors.
# Chaque executor additionne ses partitions, puis le Driver additionne les résultats partiels.
total = rdd.sum()
print("Somme calculée par le cluster :", total)

# Vérification avec la formule de Gauss : somme de 0 à n-1 = n * (n-1) / 2
n = 1_000_000
attendu = n * (n - 1) // 2
print("Somme attendue                :", attendu)
print("Résultat correct ?            :", total == attendu)

### Lire la barre de progression

Pendant l'exécution, Spark affiche une barre comme :

```
[Stage 0:>                                   (0 + 4) / 4]
```

- `Stage 0` : le groupe de tâches en cours.
- `(0 + 4) / 4` se lit : **0 terminées + 4 en cours / 4 au total**. Les 4 tâches tournent donc en parallèle, une par cœur.

Le résultat `499999500000` est bien celui de la formule de Gauss : le cluster calcule juste.

In [ ]:
# Maintenant que les executors sont démarrés, on re-vérifie l'état du cluster.
print("Cœurs (defaultParallelism) :", sc.defaultParallelism)       # doit afficher 4

# getExecutorMemoryStatus() liste toutes les JVM connectées, Driver compris : on retire 1.
nb_executors = sc._jsc.sc().getExecutorMemoryStatus().size() - 1
print("Nombre d'executors         :", nb_executors)                # attendu : 2 (un par worker)

print("Interface Spark du Driver  :", sc.uiWebUrl)                 # DAG, stages, tâches (port 4040)

# Comment le RDD a été découpé :
print("Partitions du RDD          :", rdd.getNumPartitions())
# glom() regroupe chaque partition en liste, map(len) compte ses éléments.
print("Éléments par partition     :", rdd.glom().map(len).collect())   # 4 x 250 000

### Les captures à faire pour le rapport

1. **http://localhost:8080** → *Spark Master* : **2 Workers ALIVE**, et ton application `SahelPay-Partie1` dans *Running Applications* avec **4 cœurs** utilisés.
2. **http://localhost:9001** → la console MinIO avec le bucket **`lake`**.
3. La **sortie de la cellule SparkSession** de ce notebook (version, master, appName, cœurs).
4. Facultatif mais utile : **http://localhost:4040** → onglet *Jobs* après le calcul, pour montrer les 4 tâches du Stage 0.

## 5. Vérifier MinIO et Kafka depuis Spark

Ces deux tests ne sont pas demandés pour le Jalon 1, mais ils évitent les mauvaises surprises : tout ce qu'on fera ensuite (Bronze/Silver/Gold, streaming) repose sur ces deux connexions.

### MinIO (stockage S3)
On écrit un mini-DataFrame dans le bucket `lake` puis on le relit. Le préfixe `s3a://` indique à Spark d'utiliser le protocole S3 ; l'adresse de MinIO est définie dans `spark_session.py`.

In [ ]:
# Un DataFrame de 2 lignes, créé côté Driver (les DataFrames seront vus en détail en Partie 3).
df_test = spark.createDataFrame([(1, "ping"), (2, "pong")], ["id", "message"])

# Écriture au format Parquet dans le bucket "lake" de MinIO (s3a://<bucket>/<chemin>).
# mode("overwrite") : on remplace si ça existe déjà, pour pouvoir relancer la cellule.
df_test.write.mode("overwrite").parquet("s3a://lake/tests/ping")

# Relecture depuis MinIO : on doit retrouver 2 lignes.
print("Lignes relues depuis MinIO :", spark.read.parquet("s3a://lake/tests/ping").count())

Dans la console MinIO (http://localhost:9001), le bucket `lake` contient maintenant un dossier `tests/ping` avec des fichiers `.parquet`. Tu peux le supprimer après coup, il ne sert qu'au test.

### Kafka (flux de messages)
On lit le topic `transactions` en mode *batch*. Il est vide pour l'instant, donc on attend **0 message** : l'intérêt est de vérifier que la connexion fonctionne.

In [ ]:
# Lecture "batch" du topic Kafka (en streaming on utilisera readStream en Partie 4).
df_kafka = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")   # adresse INTERNE au réseau Docker
    .option("subscribe", "transactions")                # le topic créé automatiquement
    .option("startingOffsets", "earliest")              # on lit depuis le début du topic
    .load()
)

# Le topic est vide pour l'instant : 0 est donc le résultat attendu (pas une erreur).
print("Messages dans le topic 'transactions' :", df_kafka.count())

## 6. Pour le rapport de la Partie 1

À rédiger avec tes propres mots (une page environ) :
- les rôles du **Driver**, du **Cluster Manager**, des **Executors** et du **SparkContext** ;
- le lien avec **ton déploiement** : Driver = `jupyter`, Cluster Manager = `spark-master`, Executors sur `spark-worker-1` et `spark-worker-2` ;
- les **deux limites de MapReduce** corrigées par Spark (mémoire vs disque, batch vs streaming) ;
- les **captures** listées plus haut.

## 7. Libérer le cluster avant de passer au notebook suivant

> ⚠️ Notre session réserve **tous les cœurs** du cluster (4 sur 4). Si tu ouvres un autre notebook sans libérer celui-ci, sa session restera en attente de ressources et rien ne s'exécutera.
> Avant de changer de notebook : exécute la cellule ci-dessous, ou fais *Kernel → Shut Down Kernel*.

In [ ]:
# Arrête la SparkSession et libère les cœurs et la mémoire du cluster.
# À exécuter quand on a fini avec ce notebook (la session se recrée avec get_spark()).
spark.stop()